# 16 - Task Diversity: Prefill Conditions

Run the core 2x2 matrix for each new task (MATH, ARC-Challenge, HellaSwag).
Generates paraphrases inline for each task's COTs.

In [ ]:
import subprocess, sys
from pathlib import Path

WORKSPACE = Path("/workspace/13-4-2026")
REPO_DIR = WORKSPACE / "legibility"

if REPO_DIR.exists():
    subprocess.run(["git", "-C", str(REPO_DIR), "pull"], check=True)
else:
    WORKSPACE.mkdir(parents=True, exist_ok=True)
    subprocess.run([
        "git", "clone", "-b", "13-4-2026",
        "https://github.com/JackHopkins/legibility.git",
        str(REPO_DIR)
    ], check=True)

sys.path.insert(0, str(REPO_DIR))
from lib.config import *

for d in [CACHE_DIR, COT_CACHE, PARAPHRASE_CACHE, PREFILL_CACHE, RESULTS_DIR, FIGURES_DIR]:
    d.mkdir(parents=True, exist_ok=True)

In [ ]:
import json
import gc, torch
from tqdm import tqdm
from vllm import LLM, SamplingParams
from transformers import AutoTokenizer
from lib.data import (
    load_math, load_arc_challenge, load_hellaswag,
    extract_predicted_answer_mc, extract_predicted_answer_math,
)
from lib.prefill import run_prefill_condition
from lib.paraphrase import paraphrase_cots

task_datasets = {
    "math": load_math(max_examples=500),
    "arc_challenge": load_arc_challenge(max_examples=500),
    "hellaswag": load_hellaswag(max_examples=500),
}

def load_task_cots(task_name):
    cot_dir = CACHE_DIR / f"cots_{task_name}"
    cots = {}
    for p in cot_dir.glob("*.json"):
        r = json.loads(p.read_text())
        cots[r["problem_id"]] = r["cot_text"]
    return cots

## Phase 1: Generate paraphrases for each task

In [ ]:
print(f"Loading {PARAPHRASER_MODEL} for paraphrasing...")
llm = LLM(model=PARAPHRASER_MODEL, dtype="bfloat16", max_model_len=4096)
tokenizer = AutoTokenizer.from_pretrained(PARAPHRASER_MODEL)

for task_name in task_datasets:
    cot_lookup = load_task_cots(task_name)
    if not cot_lookup:
        print(f"No COTs for {task_name}, skipping.")
        continue

    cot_list = [{"problem_id": pid, "cot_text": txt} for pid, txt in cot_lookup.items()]
    condition = f"paraphrase_light_{task_name}"
    paraphrase_cots(llm, tokenizer, cot_list, condition=condition, heavy=False)

del llm, tokenizer
gc.collect()
torch.cuda.empty_cache()

## Phase 2: Prefill with PRIMARY_MODEL (self conditions)

In [ ]:
print(f"Loading {PRIMARY_MODEL}...")
llm = LLM(model=PRIMARY_MODEL, dtype="bfloat16", max_model_len=4096)

for task_name, data in task_datasets.items():
    cot_lookup = load_task_cots(task_name)
    if not cot_lookup:
        continue

    # Load paraphrases
    para_lookup = {}
    for p in PARAPHRASE_CACHE.glob(f"paraphrase_light_{task_name}_*.json"):
        r = json.loads(p.read_text())
        para_lookup[r["problem_id"]] = r["paraphrased_cot"]

    print(f"\n=== {task_name}: self conditions ===")
    run_prefill_condition(llm, f"self_prefill_{task_name}", PRIMARY_MODEL, data, cot_lookup)
    run_prefill_condition(llm, f"paraphrase_self_{task_name}", PRIMARY_MODEL, data, para_lookup)

del llm
gc.collect()
torch.cuda.empty_cache()

## Phase 3: Prefill with CROSS_MODEL (cross conditions)

In [ ]:
print(f"Loading {CROSS_MODEL}...")
llm = LLM(model=CROSS_MODEL, dtype="bfloat16", max_model_len=4096)

for task_name, data in task_datasets.items():
    cot_lookup = load_task_cots(task_name)
    if not cot_lookup:
        continue

    para_lookup = {}
    for p in PARAPHRASE_CACHE.glob(f"paraphrase_light_{task_name}_*.json"):
        r = json.loads(p.read_text())
        para_lookup[r["problem_id"]] = r["paraphrased_cot"]

    print(f"\n=== {task_name}: cross conditions ===")
    run_prefill_condition(llm, f"cross_prefill_{task_name}", CROSS_MODEL, data, cot_lookup)
    run_prefill_condition(llm, f"paraphrase_cross_{task_name}", CROSS_MODEL, data, para_lookup)

del llm
gc.collect()
torch.cuda.empty_cache()

In [ ]:
# Summary
for task_name in task_datasets:
    print(f"\n--- {task_name} ---")
    for cond_prefix in ["self_prefill", "cross_prefill", "paraphrase_self", "paraphrase_cross", "no_cot"]:
        cond = f"{cond_prefix}_{task_name}"
        n = len(list(PREFILL_CACHE.glob(f"{cond}_*.json")))
        print(f"  {cond:40s}: {n}")